In [ ]:
## TASCA 1. Presentació del conjunt de dades

El conjunt de dades correspon a una clínica veterinària que recull informació sobre els serveis d'atenció a animals en diferents centres de Catalunya. 
L'objectiu de l'anàlisi és estudiar l'activitat veterinària, comparant els serveis, les espècies ateses, els centres, el nombre d'animals i la durada de les visites. 
També es vol detectar possibles errors o dades incompletes abans de fer l'anàlisi.

El fitxer conté 120 registres i les columnes següents:

- **servei**: tipus de servei veterinari realitzat(consulta, vacuna, radiografia, etc)
- **espècie**: espècie de l'animal atès(gos, gat, conill, ocells, exòtics...)
- **centre**: municipi o centre veterinari on s'ha realitzat l'atenció.
- **animals_atesos**: nombre d'animals atesos en el servei.
- **durada_min**: durada del servei en minuts.
- **data**: data en què es va realitzar l'atenció.

Les dades presenten alguns problemes de qualitat, com ara camps buits, espais sobrants, valors no numèrics i dates incorrectes. 
Per tant, abans de calcular estadístiques o crear gràfics, serà necessari netejar i validar el conjunt de dades.

In [ ]:
## Fases de l'anàlisi

1. **Exploració inicial:** càrrega del CSV, visualització de les primeres files, dimensions del conjunt de dades i tipus de dades.
2. **Detecció d'incidències:** identificació de camps obligatoris buits, valors numèrics incorrectes, dates invàlides i inconsistències de format.
3. **Neteja i validació:** aplicació de les transformacions necessàries i definició de les regles per determinar quins registres són vàlids.
4. **Gestió dels registres incorrectes:** identificació dels registres rebutjats i explicació dels motius pels quals no es poden acceptar.
5. **Anàlisi de les dades netes:** càlcul d’estadístiques i indicadors rellevants, amb agrupacions per les variables de classificació quan sigui convenient.
6. **Resultats i conclusions:** resum dels principals resultats i valoració de la qualitat de les dades.

In [ ]:
# EXERCICI 1 - Exploració inciial

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer: #-sig elimina el BOM
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

# Comprovacions
print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])



Camps del CSV: ['servei', 'especie', 'centre', 'animals_atesos', 'durada_min', 'data']
Nombre total de registres: 120
Exemple de primer registre: {'servei': 'Microxip', 'especie': 'Gos', 'centre': 'Girona', 'animals_atesos': '12', 'durada_min': '114,0', 'data': '2025-04-21'}


In [8]:
# EXERCICI 2 - Detecció d'indidències

def converteix_enter_positiu(valor):
    """Retorna un int positiu (>0) o None si no és vàlid."""
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def converteix_durada(valor):
    """Accepta coma o punt decimal; retorna float positiu (>0) o None."""
    try:
        v_str = str(valor).strip().replace(",", ".")
        v = float(v_str)
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def data_valida(valor: str) -> bool:
    """Retorna True si el text correspon a una data vàlida en format AAAA-MM-DD."""
    try:
        datetime.strptime(str(valor).strip(), "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False

# Inicialització de llistes per emmagatzemar els resultats
serveis_nets = []
incidencies = []
rebutjats = []